# CIFAR-10 Ensemble with 5 CNNs (Keras 3 + Torch backend)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tensorchiefs/dl_course_2025/blob/main/notebooks/cifar10_ensemble_keras3_torch.ipynb)

In [ ]:
# # Install keras with torch backend if not already available
# !pip install -q keras --upgrade
# !KERAS_BACKEND=torch python -c "import keras; print('Keras version:', keras.__version__)"


In [ ]:
import keras
keras.config.set_backend("torch")  # Ensure we're using the torch backend

import numpy as np
import matplotlib.pyplot as plt
from keras import layers, ops, Model
from keras.datasets import cifar10
from keras.utils import to_categorical
from sklearn.metrics import log_loss, accuracy_score
from tqdm import tqdm


In [ ]:
# Load and preprocess CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)


In [ ]:
# CNN model definition
def build_cnn():
    inputs = layers.Input(shape=(32, 32, 3))
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(10, activation='softmax')(x)
    model = Model(inputs, outputs)
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model


In [ ]:
# Train 5 models
models = []
histories = []
for i in range(5):
    print(f"Training model {i+1}/5")
    model = build_cnn()
    history = model.fit(x_train, y_train_cat, epochs=10, batch_size=64,
                        validation_split=0.1, verbose=2)
    models.append(model)
    histories.append(history)


In [ ]:
# Get individual model predictions
probs_list = []
for model in models:
    probs = model.predict(x_test, verbose=0)
    probs_list.append(probs)

# Average predictions
probs_ensemble = np.mean(probs_list, axis=0)

# Ensemble evaluation
y_pred_ensemble = np.argmax(probs_ensemble, axis=1)
nll_ensemble = log_loss(y_test, probs_ensemble)
acc_ensemble = accuracy_score(y_test, y_pred_ensemble)
print(f"Ensemble NLL: {nll_ensemble:.4f}")
print(f"Ensemble Accuracy: {acc_ensemble:.4f}")


In [ ]:
# Evaluate individual models
nlls = []
accs = []
for probs in probs_list:
    y_pred = np.argmax(probs, axis=1)
    nll = log_loss(y_test, probs)
    acc = accuracy_score(y_test, y_pred)
    nlls.append(nll)
    accs.append(acc)

print(f"Average Individual NLL: {np.mean(nlls):.4f}")
print(f"Average Individual Accuracy: {np.mean(accs):.4f}")


### Summary:
- This notebook trains 5 CNNs on CIFAR-10 using Keras 3 with the PyTorch backend.
- The ensemble model averages the predicted probabilities and achieves performance at least as good as the average of the individual models.
